## 3. Data Preprocessing and Feature Engineering

In [ ]:
# Prepare data for modeling
from preprocessing import prepare_data

print("=" * 80)
print("DATA PREPARATION FOR MODELING")
print("=" * 80)

with PerformanceTimer("Data Preparation"):
    X_train, X_test, y_train, y_test, preprocessor = prepare_data(
        df,
        target_col='label',
        test_size=0.2,
        random_state=RANDOM_STATE,
        scale_features=True
    )

print(f"\n{'='*80}")
print("TRAIN-TEST SPLIT SUMMARY")
print(f"{'='*80}")
print(f"\nTraining Set:")
print(f"  Samples: {len(X_train):,}")
print(f"  Shape: {X_train.shape}")
print(f"  Classes: {len(np.unique(y_train))}")

print(f"\nTest Set:")
print(f"  Samples: {len(X_test):,}")
print(f"  Shape: {X_test.shape}")
print(f"  Classes: {len(np.unique(y_test))}")

print(f"\nPreprocessor Configuration:")
config = preprocessor.get_config()
for key, value in config.items():
    if key not in ['classes', 'feature_names']:
        print(f"  {key}: {value}")

# Verify class distribution in splits
train_dist = pd.Series(y_train).value_counts().sort_index()
test_dist = pd.Series(y_test).value_counts().sort_index()

print(f"\nClass Distribution Verification:")
print(f"  Training: {train_dist.min()}-{train_dist.max()} samples per class")
print(f"  Test: {test_dist.min()}-{test_dist.max()} samples per class")
print(f"  ✓ Stratified split maintains class balance")

## 4. Model Training and Cross-Validation

In [ ]:
# Initialize model trainer
from training import ModelTrainer

print("=" * 80)
print("MODEL TRAINING WITH CROSS-VALIDATION")
print("=" * 80)

trainer = ModelTrainer(random_state=RANDOM_STATE, n_jobs=-1)

# Train models with 5-fold stratified cross-validation
CV_FOLDS = 5

with PerformanceTimer("Cross-Validation Training"):
    cv_results = trainer.train_with_cv(X_train, y_train, cv=CV_FOLDS)

print(f"\n{'='*80}")
print("CROSS-VALIDATION RESULTS")
print(f"{'='*80}")
print(f"\nFolds: {CV_FOLDS}")
print(f"Scoring: Accuracy\n")

# Create results DataFrame
cv_summary = pd.DataFrame([
    {
        'Model': name,
        'CV Mean': results['cv_mean'],
        'CV Std': results['cv_std'],
        'Training Time (s)': results['training_time']
    }
    for name, results in cv_results.items()
]).sort_values('CV Mean', ascending=False)

print(cv_summary.to_string(index=False))

# Visualize CV results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: CV Accuracy with error bars
models = cv_summary['Model'].tolist()
means = cv_summary['CV Mean'].tolist()
stds = cv_summary['CV Std'].tolist()

x_pos = np.arange(len(models))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(models)))

ax1.bar(x_pos, means, yerr=stds, align='center', alpha=0.8, 
        color=colors, ecolor='black', capsize=10)
ax1.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax1.set_xlabel('Model', fontsize=12, fontweight='bold')
ax1.set_title(f'{CV_FOLDS}-Fold Cross-Validation Accuracy', fontsize=14, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(models, rotation=45, ha='right')
ax1.set_ylim([0.90, 1.01])
ax1.grid(True, alpha=0.3, axis='y')

# Add value labels
for i, (mean, std) in enumerate(zip(means, stds)):
    ax1.text(i, mean + std + 0.002, f'{mean:.4f}', 
             ha='center', va='bottom', fontsize=9, fontweight='bold')

# Plot 2: Training time comparison
times = cv_summary['Training Time (s)'].tolist()
ax2.barh(x_pos, times, align='center', alpha=0.8, color=colors)
ax2.set_xlabel('Training Time (seconds)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Model', fontsize=12, fontweight='bold')
ax2.set_title('Training Time Comparison', fontsize=14, fontweight='bold')
ax2.set_yticks(x_pos)
ax2.set_yticklabels(models)
ax2.grid(True, alpha=0.3, axis='x')

# Add value labels
for i, time in enumerate(times):
    ax2.text(time + 0.5, i, f'{time:.2f}s', 
             ha='left', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/cv_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved: reports/figures/cv_results.png")

## 5. Hyperparameter Optimization

In [ ]:
# Hyperparameter tuning for top 3 models
print("=" * 80)
print("HYPERPARAMETER OPTIMIZATION")
print("=" * 80)

# Select top 3 models
top_3_models = cv_summary['Model'].head(3).tolist()
print(f"\nTuning models: {', '.join(top_3_models)}")
print(f"Search method: RandomizedSearchCV")
print(f"Iterations: 50 per model")
print(f"Cross-validation folds: {CV_FOLDS}\n")

with PerformanceTimer("Hyperparameter Tuning"):
    best_params = trainer.hyperparameter_tuning(
        X_train,
        y_train,
        model_names=top_3_models,
        n_iter=50,
        cv=CV_FOLDS
    )

print(f"\n{'='*80}")
print("OPTIMIZED HYPERPARAMETERS")
print(f"{'='*80}\n")

for model_name, params in best_params.items():
    print(f"{model_name}:")
    for param, value in params.items():
        print(f"  {param}: {value}")
    print()

# Compare before and after tuning
print("Performance Improvement:")
print("-" * 60)
for model_name in top_3_models:
    before = cv_results[model_name]['cv_mean']
    after = cv_results[model_name].get('best_cv_score', before)
    improvement = (after - before) * 100
    print(f"{model_name:20s}: {before:.4f} → {after:.4f} (+{improvement:.2f}%)")

## 6. Model Evaluation on Test Set

In [ ]:
# Evaluate all models on test set
print("=" * 80)
print("TEST SET EVALUATION")
print("=" * 80)

with PerformanceTimer("Test Set Evaluation"):
    eval_results = trainer.evaluate_models(X_test, y_test)

# Create evaluation summary
eval_summary = pd.DataFrame([
    {
        'Model': name,
        'Accuracy': results['accuracy'],
        'Precision': results['precision'],
        'Recall': results['recall'],
        'F1 Score': results['f1_score']
    }
    for name, results in eval_results.items()
]).sort_values('Accuracy', ascending=False)

print(f"\n{'='*80}")
print("TEST SET PERFORMANCE METRICS")
print(f"{'='*80}\n")
print(eval_summary.to_string(index=False))

# Visualize test performance
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(eval_summary))
width = 0.2

metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    values = eval_summary[metric].values
    ax.bar(x + i * width, values, width, label=metric, alpha=0.8, color=color)

ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Test Set Performance - All Metrics', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(eval_summary['Model'], rotation=45, ha='right')
ax.set_ylim([0.90, 1.01])
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../reports/figures/test_performance.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved: reports/figures/test_performance.png")

# Save evaluation results
eval_summary.to_csv('../reports/model_evaluation.csv', index=False)
print("✓ Results saved: reports/model_evaluation.csv")

## 7. Best Model Selection and Analysis

In [ ]:
# Select best model
best_name, best_model = trainer.select_best_model(eval_results, metric='accuracy')

print("=" * 80)
print("BEST MODEL SELECTION")
print("=" * 80)

print(f"\n🏆 Best Model: {best_name}\n")

# Detailed performance
best_results = eval_results[best_name]
print("Performance Metrics:")
print("-" * 60)
print(f"  Accuracy:  {best_results['accuracy']:.6f} ({best_results['accuracy']*100:.2f}%)")
print(f"  Precision: {best_results['precision']:.6f} ({best_results['precision']*100:.2f}%)")
print(f"  Recall:    {best_results['recall']:.6f} ({best_results['recall']*100:.2f}%)")
print(f"  F1 Score:  {best_results['f1_score']:.6f} ({best_results['f1_score']*100:.2f}%)")

# Cross-validation performance
print(f"\nCross-Validation:")
print("-" * 60)
print(f"  CV Mean:   {cv_results[best_name]['cv_mean']:.6f}")
print(f"  CV Std:    {cv_results[best_name]['cv_std']:.6f}")

# Overfitting check
cv_acc = cv_results[best_name]['cv_mean']
test_acc = best_results['accuracy']
diff = abs(cv_acc - test_acc)

print(f"\nGeneralization Analysis:")
print("-" * 60)
print(f"  CV Accuracy:   {cv_acc:.6f}")
print(f"  Test Accuracy: {test_acc:.6f}")
print(f"  Difference:    {diff:.6f} ({diff*100:.2f}%)")

if diff < 0.01:
    print(f"  Status: ✓ Excellent generalization (diff < 1%)")
elif diff < 0.03:
    print(f"  Status: ✓ Good generalization (diff < 3%)")
else:
    print(f"  Status: ⚠ Possible overfitting (diff >= 3%)")

## 8. Confusion Matrix Analysis

In [ ]:
# Plot confusion matrix for best model
from sklearn.metrics import ConfusionMatrixDisplay

print("=" * 80)
print("CONFUSION MATRIX ANALYSIS")
print("=" * 80)

cm = best_results['confusion_matrix']
y_pred = best_results['predictions']

# Convert encoded labels back to crop names
crop_names = preprocessor.classes_
y_test_names = preprocessor.inverse_transform_target(y_test)
y_pred_names = preprocessor.inverse_transform_target(y_pred)

# Create figure
fig, ax = plt.subplots(figsize=(16, 14))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=crop_names
)

disp.plot(ax=ax, cmap='Blues', values_format='d', xticks_rotation=45)
ax.set_title(f'Confusion Matrix - {best_name}', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Predicted Crop', fontsize=12, fontweight='bold')
ax.set_ylabel('True Crop', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved: reports/figures/confusion_matrix.png")

# Analyze misclassifications
misclassified = y_test != y_pred
n_misclassified = misclassified.sum()

print(f"\nMisclassification Analysis:")
print("-" * 60)
print(f"  Total test samples: {len(y_test)}")
print(f"  Correctly classified: {len(y_test) - n_misclassified}")
print(f"  Misclassified: {n_misclassified}")
print(f"  Error rate: {(n_misclassified/len(y_test))*100:.2f}%")

if n_misclassified > 0:
    print(f"\nMisclassified Samples:")
    for i, (true_label, pred_label) in enumerate(zip(y_test_names[misclassified], 
                                                       y_pred_names[misclassified])):
        if i < 5:  # Show first 5
            print(f"  Sample {i+1}: True={true_label}, Predicted={pred_label}")
    if n_misclassified > 5:
        print(f"  ... and {n_misclassified - 5} more")

## 9. Feature Importance Analysis

In [ ]:
# Get feature importance
print("=" * 80)
print("FEATURE IMPORTANCE ANALYSIS")
print("=" * 80)

feature_names = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
importance_df = trainer.get_feature_importance(
    model_name=best_name,
    feature_names=feature_names
)

if importance_df is not None:
    print(f"\nFeature Importance Ranking ({best_name}):")
    print("-" * 60)
    
    for idx, row in importance_df.iterrows():
        bar = '█' * int(row['importance'] * 50)
        print(f"  {idx+1}. {row['feature']:15s}: {row['importance']:.4f} {bar}")
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 6))
    
    colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(importance_df)))
    bars = ax.barh(importance_df['feature'], importance_df['importance'], 
                   color=colors, alpha=0.8, edgecolor='black')
    
    ax.set_xlabel('Importance', fontsize=12, fontweight='bold')
    ax.set_ylabel('Feature', fontsize=12, fontweight='bold')
    ax.set_title(f'Feature Importance - {best_name}', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    
    # Add value labels
    for i, (bar, importance) in enumerate(zip(bars, importance_df['importance'])):
        ax.text(importance + 0.01, bar.get_y() + bar.get_height()/2,
                f'{importance:.4f}', va='center', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('../reports/figures/feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n✓ Figure saved: reports/figures/feature_importance.png")
    
    # Save to CSV
    importance_df.to_csv('../reports/feature_importance.csv', index=False)
    print("✓ Data saved: reports/feature_importance.csv")
    
    # Insights
    print(f"\nKey Insights:")
    print("-" * 60)
    top_3 = importance_df.head(3)
    top_3_importance = top_3['importance'].sum()
    print(f"  Top 3 features account for {top_3_importance*100:.1f}% of importance")
    print(f"  Most important: {top_3.iloc[0]['feature']} ({top_3.iloc[0]['importance']*100:.1f}%)")
    print(f"  Least important: {importance_df.iloc[-1]['feature']} ({importance_df.iloc[-1]['importance']*100:.1f}%)")
else:
    print(f"\n⚠ {best_name} does not support feature importance extraction")